In [ ]:
# saving the calibration data 




In [1]:
import csv
import numpy as np
from nuscenes.nuscenes import NuScenes
from pyquaternion import Quaternion
from tqdm import tqdm

def get_lidar2img_from_nuscenes(nusc, sample_token, cam_channel="CAM_FRONT"):
    sample = nusc.get("sample", sample_token)
    sd_lidar = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
    sd_cam = nusc.get("sample_data", sample["data"][cam_channel])
    cs_lidar = nusc.get("calibrated_sensor", sd_lidar["calibrated_sensor_token"])
    cs_cam = nusc.get("calibrated_sensor", sd_cam["calibrated_sensor_token"])
    ep_lidar = nusc.get("ego_pose", sd_lidar["ego_pose_token"])
    ep_cam = nusc.get("ego_pose", sd_cam["ego_pose_token"])
    lidar2ego = np.eye(4)
    lidar2ego[:3, :3] = Quaternion(cs_lidar["rotation"]).rotation_matrix
    lidar2ego[:3, 3] = cs_lidar["translation"]
    ego_lidar2global = np.eye(4)
    ego_lidar2global[:3, :3] = Quaternion(ep_lidar["rotation"]).rotation_matrix
    ego_lidar2global[:3, 3] = ep_lidar["translation"]
    global2ego_cam = np.eye(4)
    global2ego_cam[:3, :3] = Quaternion(ep_cam["rotation"]).rotation_matrix.T
    global2ego_cam[:3, 3] = -global2ego_cam[:3, :3].dot(ep_cam["translation"])
    ego_cam2cam = np.eye(4)
    ego_cam2cam[:3, :3] = Quaternion(cs_cam["rotation"]).rotation_matrix.T
    ego_cam2cam[:3, 3] = -ego_cam2cam[:3, :3].dot(cs_cam["translation"])
    lidar2cam = ego_cam2cam @ global2ego_cam @ ego_lidar2global @ lidar2ego
    K = np.array(cs_cam["camera_intrinsic"])
    lidar2img = K @ lidar2cam[:3, :]
    R = Quaternion(cs_cam["rotation"]).rotation_matrix
    T = np.array(cs_cam["translation"])
    img_shape = (sd_cam["height"], sd_cam["width"])
    return R, T, K, lidar2img, img_shape

def save_calib_to_csv_front_cam(nusc, out_csv="calib_trainval_front.csv"):
    cam_channel = "CAM_FRONT"
    all_samples = nusc.sample
    with open(out_csv, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "sample_token", "cam_channel",
            "K_flat", "R_flat", "T_flat",
            "lidar2img_flat", "img_height", "img_width"
        ])
        for sample in tqdm(all_samples, desc="Saving calibration for CAM_FRONT"):
            token = sample["token"]
            try:
                R, T, K, lidar2img, img_shape = get_lidar2img_from_nuscenes(nusc, token, cam_channel)
                writer.writerow([
                    token, cam_channel,
                    K.flatten().tolist(),
                    R.flatten().tolist(),
                    T.flatten().tolist(),
                    lidar2img.flatten().tolist(),
                    img_shape[0], img_shape[1]
                ])
            except Exception as e:
                print(f"[WARNING] Skipping {token}: {e}")
    print(f"[INFO] CAM_FRONT calibration info saved to {out_csv}")

# --- Run this script ---

dataroot = "/home/draiman/Desktop/datasets/nuscenes"
nusc = NuScenes(version="v1.0-trainval", dataroot=dataroot, verbose=False)
save_calib_to_csv_front_cam(nusc, "calib_trainval_front.csv")


Saving calibration for CAM_FRONT: 100%|██████████| 34149/34149 [00:03<00:00, 8595.95it/s]

[INFO] CAM_FRONT calibration info saved to calib_trainval_front.csv


In [ ]:
# import csv
# from nuscenes.nuscenes import NuScenes
# from tqdm import tqdm

# def save_token_to_file_csv(nusc, out_csv="token_to_file.csv"):
#     cam_channel = "CAM_FRONT"
#     lidar_channel = "LIDAR_TOP"
#     all_samples = nusc.sample

#     with open(out_csv, mode='w', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow(["sample_token", "cam_channel", "image_filename", "lidar_filename"])

#         for sample in tqdm(all_samples, desc="Saving token-file mapping"):
#             token = sample["token"]
#             try:
#                 # Camera image
#                 sd_cam = nusc.get("sample_data", sample["data"][cam_channel])
#                 image_filename = sd_cam["filename"]  # Usually something like "samples/CAM_FRONT/....jpg"
#                 # LiDAR point cloud
#                 sd_lidar = nusc.get("sample_data", sample["data"][lidar_channel])
#                 lidar_filename = sd_lidar["filename"]  # Usually something like "samples/LIDAR_TOP/....pcd.bin"
#                 writer.writerow([token, cam_channel, image_filename, lidar_filename])
#             except Exception as e:
#                 print(f"[WARNING] Skipping {token}: {e}")
#     print(f"[INFO] Token-to-file mapping saved to {out_csv}")

# # --- Run this script ---

# dataroot = "/home/draiman/Desktop/datasets/nuscenes"
# nusc = NuScenes(version="v1.0-trainval", dataroot=dataroot, verbose=False)
# save_token_to_file_csv(nusc, "token_to_file.csv")


In [ ]:
# testing reading a sample's lidar and camera filenames
from nuscenes.nuscenes import NuScenes

dataroot = "/home/draiman/Desktop/datasets/nuscenes"
nusc = NuScenes(version="v1.0-trainval", dataroot=dataroot, verbose=False)

sample = nusc.sample[0]
cam_channel = "CAM_FRONT"
lidar_channel = "LIDAR_TOP"

# Get tokens and filenames
sample_token = sample["token"]
sd_cam = nusc.get("sample_data", sample["data"][cam_channel])
sd_lidar = nusc.get("sample_data", sample["data"][lidar_channel])

image_filename = sd_cam["filename"]    # e.g. samples/CAM_FRONT/n008-2018-08-01-15-16-36-0400__CAM_FRONT__1533151614732460.jpg
lidar_filename = sd_lidar["filename"]  # e.g. samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151614757653.pcd.bin

print("Sample token:", sample_token)
print("Camera image filename:", image_filename)
print("LiDAR filename:", lidar_filename)


Sample token: e93e98b63d3b40209056d129dc53ceee
Camera image filename: samples/CAM_FRONT/n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883530412470.jpg
LiDAR filename: samples/LIDAR_TOP/n015-2018-07-18-11-07-57+0800__LIDAR_TOP__1531883530449377.pcd.bin


In [ ]:
# testing if i can take the samples and  
import os

validation_folder = "/home/draiman/Desktop/Datasets_nuscenes/validation/lidar"
validation_tokens = set(
    os.path.splitext(f)[0]
    for f in os.listdir(validation_folder)
    if f.endswith(".pt")
)

from nuscenes.nuscenes import NuScenes
from tqdm import tqdm

dataroot = "/home/draiman/Desktop/datasets/nuscenes"
nusc = NuScenes(version="v1.0-trainval", dataroot=dataroot, verbose=False)

cam_channel = "CAM_FRONT"
lidar_channel = "LIDAR_TOP"

for sample in tqdm(nusc.sample[:5]):  # Use [:5] just to demo, remove for all
    token = sample["token"]
    sd_cam = nusc.get("sample_data", sample["data"][cam_channel])
    sd_lidar = nusc.get("sample_data", sample["data"][lidar_channel])
    image_filename = sd_cam["filename"]
    lidar_filename = sd_lidar["filename"]
    split_type = "validation" if token in validation_tokens else "train"

    print(f"{token},{cam_channel},{image_filename},{lidar_filename},{split_type}")



100%|██████████| 5/5 [00:00<00:00, 50051.36it/s]

e93e98b63d3b40209056d129dc53ceee,CAM_FRONT,samples/CAM_FRONT/n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883530412470.jpg,samples/LIDAR_TOP/n015-2018-07-18-11-07-57+0800__LIDAR_TOP__1531883530449377.pcd.bin,train
14d5adfe50bb4445bc3aa5fe607691a8,CAM_FRONT,samples/CAM_FRONT/n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883530912460.jpg,samples/LIDAR_TOP/n015-2018-07-18-11-07-57+0800__LIDAR_TOP__1531883530949817.pcd.bin,train
ae4e0c3aa3f24c91aab599e8b54e9264,CAM_FRONT,samples/CAM_FRONT/n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883531412477.jpg,samples/LIDAR_TOP/n015-2018-07-18-11-07-57+0800__LIDAR_TOP__1531883531450214.pcd.bin,train
8de7ec06e1ac48c689c4d24d6cc64fd7,CAM_FRONT,samples/CAM_FRONT/n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883531912467.jpg,samples/LIDAR_TOP/n015-2018-07-18-11-07-57+0800__LIDAR_TOP__1531883531950107.pcd.bin,train
ba94cb79ebc74614bc2442185cb53c26,CAM_FRONT,samples/CAM_FRONT/n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883532412464.jpg,samples/LIDAR_TO

In [6]:
import os
import csv
from nuscenes.nuscenes import NuScenes
from tqdm import tqdm

# 1. Build set of validation tokens from your validation folder
validation_folder = "/home/draiman/Desktop/Datasets_nuscenes/validation/lidar"  # <--- update this path
validation_tokens = set(
    os.path.splitext(f)[0]
    for f in os.listdir(validation_folder)
    if f.endswith(".pt")
)

# 2. Setup nuScenes
dataroot = "/home/draiman/Desktop/datasets/nuscenes"
nusc = NuScenes(version="v1.0-trainval", dataroot=dataroot, verbose=False)
cam_channel = "CAM_FRONT"
lidar_channel = "LIDAR_TOP"

# 3. Write mapping to CSV
output_csv = "token_image_lidar_split.csv"
with open(output_csv, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["sample_token", "cam_channel", "image_filename", "lidar_filename", "split"])
    for sample in tqdm(nusc.sample, desc="Saving token-file mapping with split"):
        token = sample["token"]
        try:
            sd_cam = nusc.get("sample_data", sample["data"][cam_channel])
            sd_lidar = nusc.get("sample_data", sample["data"][lidar_channel])
            image_filename = sd_cam["filename"]
            lidar_filename = sd_lidar["filename"]
            split_type = "validation" if token in validation_tokens else "train"
            writer.writerow([token, cam_channel, image_filename, lidar_filename, split_type])
        except Exception as e:
            print(f"[WARNING] Skipping {token}: {e}")

print(f"[INFO] Mapping with split saved to {output_csv}")


Saving token-file mapping with split: 100%|██████████| 34149/34149 [00:00<00:00, 169900.98it/s]

[INFO] Mapping with split saved to token_image_lidar_split.csv


In [ ]:
import torch

# Path to your saved feature file
feature_path = '/data/nusc_lidar/train/dee909131941447b98da1f253c64c698.pt'  # <-- update with your file

# Load the file
data = torch.load(feature_path, map_location='cpu')

# Print the meta dictionary
print("Meta info:")
print(data.keys())
print(data['meta'])


Meta info:
dict_keys(['voxel_feats', 'spatial', 'bev_c1', 'bev_c2', 'meta'])
{'sample_token': 'dee909131941447b98da1f253c64c698', 'lidar_filename': 'n008-2018-05-21-11-06-59-0400__LIDAR_TOP__1526915243547836.pcd.bin', 'split': 'train'}
